# Baseline vs. reranking RAG

This notebook compares dense retrieval and cross-encoder reranking over one versioned PDF. It runs locally or as a standalone Kaggle Notebook.

Source: Leder Games, [*The Law of Root*](https://ledergames.com/pages/resources) (October 13, 2025). The exact PDF is checked by SHA-256 below.

## 0. Kaggle setup

In Kaggle, enable Internet in Session options so the notebook can install packages, download the official PDF, and load the public retrieval models. A GPU is optional. For answer generation, add a Kaggle Secret named `GROQ_API_KEY`. The key is never printed or stored in the notebook.

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path('/kaggle/working').is_dir():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'pypdf>=6.9,<7', 'PyMuPDF>=1.25,<2', 'sentence-transformers>=5,<6',
        'faiss-cpu>=1.9,<2', 'groq>=1,<2', 'pandas>=2,<4'])
    print('Kaggle dependencies ready')
else:
    print('Local run: install requirements.txt before running the notebook')

## 1. Source validation

Locally, place the PDF at `data/law_of_root_oct_2025.pdf`. In Kaggle, the notebook downloads the official copy to `/kaggle/working` when needed. The checksum stops the run if its contents differ from the documented version.

In [ ]:
from hashlib import sha256
from urllib.request import urlretrieve

from pypdf import PdfReader

PDF_NAME = 'law_of_root_oct_2025.pdf'
PDF_URL = 'https://cdn.shopify.com/s/files/1/0106/0162/7706/files/Root_Base_Law_Oct_2025.pdf?v=1774903756'
EXPECTED_SHA256 = 'd32c8c0681d5bac6d2b483fb75d77efb47e78deaa36cb798bc527a605909c045'

if Path('/kaggle/working').is_dir():
    PDF_PATH = Path('/kaggle/working') / PDF_NAME
    if not PDF_PATH.is_file():
        print('Downloading the official rulebook...')
        urlretrieve(PDF_URL, PDF_PATH)
else:
    PDF_PATH = Path('data') / PDF_NAME

if not PDF_PATH.is_file():
    raise FileNotFoundError(f'Download the official rulebook to {PDF_PATH}')

actual_sha256 = sha256(PDF_PATH.read_bytes()).hexdigest()
if actual_sha256 != EXPECTED_SHA256:
    raise ValueError(f'Unexpected PDF version: SHA-256 is {actual_sha256}')

reader = PdfReader(PDF_PATH)
page_text = [page.extract_text() or '' for page in reader.pages]
if any(not text.strip() for text in page_text):
    raise ValueError('At least one PDF page has no extractable text')

print(f'Source verified: {PDF_PATH.name}')
print(f'Pages: {len(page_text)} | Extracted characters: {sum(map(len, page_text)):,}')
print(f'SHA-256: {actual_sha256}')

## 2. Extract rules in reading order

The PDF stores some right-column text before left-column text. We sort text blocks by column and then vertical position on each page. The corpus includes printed pages 2–29; the cover, credits/index, and repeated quick reference are omitted. Page numbers remain attached to every block.

In [ ]:
import re
import fitz

RULE_PAGES = range(2, 30)  # printed page numbers, inclusive of 2–29
RULE_REF = re.compile(r'^(?P<ref>(?:\d+\.(?:\d+(?:\.\d+)*)?|[A-Z]\.(?:\d+(?:\.\d+)*)?))\s+(?=[A-Z])')
HEADING_LINE = re.compile(r'(?m)(?=^(?:\d+\.(?:\d+(?:\.\d+)*)?|[A-Z]\.(?:\d+(?:\.\d+)*)?)[ \t]+[A-Z])')

def clean_block(raw_text):
    # Soft hyphens mark words broken by PDF line wraps. Keep genuine hyphens.
    text = raw_text.replace('\u00ad\n', '').replace('\u00ad', '')
    text = re.sub(r'(?<=[a-z])-\s*\n(?=[a-z])', '', text)
    text = re.sub(r'-\s*\n(?=[A-Z])', '-', text)
    # Normalize typographic punctuation; keep the rule's actual words intact.
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = text.replace('\u2013', '-').replace('\u2014', '-')
    return re.sub(r'\s+', ' ', text).strip()

source_units = []
with fitz.open(PDF_PATH) as pdf:
    for printed_page in RULE_PAGES:
        page = pdf[printed_page - 1]
        midpoint = page.rect.width / 2
        blocks = [b for b in page.get_text('blocks')
                  if b[6] == 0 and b[1] < 750 and b[3] > 30]
        blocks.sort(key=lambda b: (b[0] >= midpoint, b[1], b[0]))
        for block in blocks:
            column = 'right' if block[0] >= midpoint else 'left'
            # A heading can share one PDF block with the previous paragraph.
            for raw_part in HEADING_LINE.split(block[4]):
                cleaned = clean_block(raw_part)
                if not cleaned:
                    continue
                joins_next = raw_part.rstrip().endswith('\u00ad')
                if (source_units and source_units[-1]['joins_next']
                        and cleaned[:1].islower()):
                    # PDF text blocks sometimes split a hyphenated word.
                    source_units[-1]['text'] += cleaned
                    source_units[-1]['page_end'] = printed_page
                    source_units[-1]['joins_next'] = joins_next
                    continue
                match = RULE_REF.match(cleaned)
                source_units.append({
                    'page_start': printed_page,
                    'page_end': printed_page,
                    'column': column,
                    'rule_ref': match.group('ref').rstrip('.') if match else None,
                    'text': cleaned,
                    'joins_next': joins_next,
                })

assert source_units and {p for u in source_units for p in range(u['page_start'], u['page_end'] + 1)} == set(RULE_PAGES)
assert all('\u00ad' not in u['text'] and '\ufffd' not in u['text'] for u in source_units)
assert any(u['rule_ref'] == '1.5.5' and 'player or their pieces' in u['text'] for u in source_units)
assert any(u['rule_ref'] == '2' and u['text'] == '2. Key Concepts' for u in source_units)
print(f'Rule pages: {len(RULE_PAGES)} | Text units: {len(source_units):,}')
print('First units:', [(u['page_start'], u['column'], u['text'][:65]) for u in source_units[:4]])

## 3. Create one shared chunk corpus

Chunks keep whole PDF text blocks where possible and overlap with neighboring blocks. A chunk records every page and rule reference it contains. Both retrieval pipelines will use this exact `chunks` list. Word counts are provisional until the embedding model's tokenizer is available.

In [ ]:
TARGET_WORDS = 260
MAX_WORDS = 320
OVERLAP_WORDS = 60

def word_count(unit):
    return len(unit['text'].split())

def build_chunks(units):
    result = []
    start = 0
    while start < len(units):
        end = start
        words = 0
        while end < len(units):
            next_words = word_count(units[end])
            if end > start and words + next_words > MAX_WORDS:
                break
            if words >= TARGET_WORDS and units[end]['rule_ref']:
                break
            words += next_words
            end += 1
        selected = units[start:end]
        refs = list(dict.fromkeys(u['rule_ref'] for u in selected if u['rule_ref']))
        result.append({
            'chunk_id': f'chunk_{len(result):04d}',
            'text': '\n'.join(u['text'] for u in selected),
            'page_start': selected[0]['page_start'],
            'page_end': selected[-1]['page_end'],
            'rule_refs': refs,
            'word_count': words,
            'source_block_start': start,
            'source_block_end': end - 1,
        })
        if end == len(units):
            break
        # Reuse trailing complete blocks as overlap. A long block is not split.
        next_start = end
        overlap = 0
        while next_start > start and overlap < OVERLAP_WORDS:
            next_start -= 1
            overlap += word_count(units[next_start])
        start = next_start if next_start > start else end
    return result

chunks = build_chunks(source_units)
assert len({c['chunk_id'] for c in chunks}) == len(chunks)
assert all(c['text'].strip() and c['page_start'] <= c['page_end'] for c in chunks)
assert all(c['word_count'] <= MAX_WORDS for c in chunks)
assert chunks[0]['source_block_start'] == 0
assert chunks[-1]['source_block_end'] == len(source_units) - 1
assert all(a['source_block_end'] + 1 >= b['source_block_start']
           for a, b in zip(chunks, chunks[1:]))

sizes = sorted(c['word_count'] for c in chunks)
print(f'Chunks: {len(chunks)} | Words: min={sizes[0]}, median={sizes[len(sizes)//2]}, max={sizes[-1]}')
print('Sample:', {k: chunks[0][k] for k in ('chunk_id', 'page_start', 'page_end', 'rule_refs', 'word_count')})
print(chunks[0]['text'][:700])

## 4. Bi-encoder and cosine index

The embedding model has a finite input length. Rebuild the word-based chunks at smaller sizes until every complete chunk fits its tokenizer, then index normalized embeddings in FAISS. This final `chunks` list is shared by both pipelines. Model loading and indexing are timed separately from per-query latency.

In [ ]:
from time import perf_counter, sleep
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder

BI_ENCODER_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
CROSS_ENCODER_NAME = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

model_load_start = perf_counter()
bi_encoder = SentenceTransformer(BI_ENCODER_NAME)
bi_model_load_seconds = perf_counter() - model_load_start

def chunk_token_lengths(items):
    return [len(bi_encoder.tokenizer(c['text'], add_special_tokens=True, truncation=False,
                                     verbose=False)['input_ids'])
            for c in items]

for _ in range(8):
    token_lengths = chunk_token_lengths(chunks)
    if max(token_lengths) <= bi_encoder.max_seq_length:
        break
    MAX_WORDS = max(100, int(MAX_WORDS * 0.8))
    TARGET_WORDS = min(TARGET_WORDS, int(MAX_WORDS * 0.8))
    OVERLAP_WORDS = min(OVERLAP_WORDS, int(MAX_WORDS * 0.25))
    chunks = build_chunks(source_units)
else:
    raise ValueError('A source block is too long for the embedding model; split it before indexing')

assert len(chunks) >= 10 and len({c['chunk_id'] for c in chunks}) == len(chunks)
print(f'Embedding limit: {bi_encoder.max_seq_length} tokens | Largest chunk: {max(token_lengths)} tokens')
print(f'Final corpus: {len(chunks)} chunks | Maximum words: {MAX_WORDS} | Overlap: {OVERLAP_WORDS} words')

index_start = perf_counter()
embeddings = bi_encoder.encode([c['text'] for c in chunks], batch_size=32,
                               convert_to_numpy=True, normalize_embeddings=True,
                               show_progress_bar=False)
embeddings = np.ascontiguousarray(embeddings, dtype=np.float32)
vector_index = faiss.IndexFlatIP(embeddings.shape[1])
vector_index.add(embeddings)
index_seconds = perf_counter() - index_start
assert vector_index.ntotal == len(chunks)
print(f'Index dimensions: {embeddings.shape[1]} | Indexing: {index_seconds:.2f}s | Model load: {bi_model_load_seconds:.2f}s')

## 5. Baseline top 10 and cross-encoder reranking

Baseline retrieval takes the first three of ten cosine-ranked candidates. Reranking scores those same ten query/chunk pairs and takes its first three. The cross-encoder never searches the full corpus.

In [ ]:
cross_load_start = perf_counter()
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME)
cross_model_load_seconds = perf_counter() - cross_load_start

def retrieve_top10(query):
    t0 = perf_counter()
    query_vector = bi_encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    query_vector = np.ascontiguousarray(query_vector, dtype=np.float32)
    embedding_seconds = perf_counter() - t0
    t1 = perf_counter()
    scores, positions = vector_index.search(query_vector, 10)
    search_seconds = perf_counter() - t1
    candidates = []
    for rank, (position, score) in enumerate(zip(positions[0], scores[0]), start=1):
        candidates.append({**chunks[int(position)], 'baseline_rank': rank,
                           'cosine_score': float(score)})
    return candidates, {'query_embedding_s': embedding_seconds, 'vector_search_s': search_seconds}

def rerank_top10(query, candidates):
    assert len(candidates) == 10
    pairs = [(query, candidate['text']) for candidate in candidates]
    t0 = perf_counter()
    scores = cross_encoder.predict(pairs, batch_size=10, show_progress_bar=False)
    scoring_seconds = perf_counter() - t0
    reranked = [{**candidate, 'cross_score': float(score)}
                for candidate, score in zip(candidates, scores)]
    reranked.sort(key=lambda item: (-item['cross_score'], item['baseline_rank']))
    for rank, candidate in enumerate(reranked, start=1):
        candidate['reranked_rank'] = rank
    return reranked, {'cross_encoder_s': scoring_seconds}

def compare_retrieval(query):
    baseline, baseline_timing = retrieve_top10(query)
    reranked, rerank_timing = rerank_top10(query, baseline)
    assert {c['chunk_id'] for c in baseline} == {c['chunk_id'] for c in reranked}
    return {'query': query, 'baseline_top10': baseline, 'reranked_top10': reranked,
            'baseline_top3': baseline[:3], 'reranked_top3': reranked[:3],
            'timing': {**baseline_timing, **rerank_timing}}

print(f'Cross-encoder loaded in {cross_model_load_seconds:.2f}s')

## 6. Identical answer prompt for both pipelines

Groq generation runs when `GROQ_API_KEY` is available as a Kaggle Secret or environment variable. Both answers use the same model, prompt template, sampling setting, and output limit; only the top-three excerpts differ. The export records finish reasons and token usage so incomplete answers are visible.

In [ ]:
import os
from groq import Groq

GROQ_MODEL = 'qwen/qwen3.8-27b'
GROQ_MAX_OUTPUT_TOKENS = 1024
PROMPT_TEMPLATE = (
    'Answer the question using only the document excerpts below. Include relevant restrictions, '
    'exceptions, and conditions. If the excerpts do not contain enough information, say so. '
    'Cite supporting chunk IDs and page numbers.\n\nQuestion: {query}\n\nDocument excerpts:\n{context}'
)

def format_context(selected):
    return '\n\n'.join(
        f"[{c['chunk_id']}, pages {c['page_start']}-{c['page_end']}]\n{c['text']}"
        for c in selected
    )

def generate_answer(client, query, selected):
    prompt = PROMPT_TEMPLATE.format(query=query, context=format_context(selected))
    t0 = perf_counter()
    response = client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        reasoning_effort='none', temperature=0.2,
        max_completion_tokens=GROQ_MAX_OUTPUT_TOKENS,
    )
    elapsed = perf_counter() - t0
    choice = response.choices[0] if response.choices else None
    usage = response.usage
    token_details = getattr(usage, 'completion_tokens_details', None)
    return {
        'answer': (choice.message.content or '') if choice else '', 'llm_s': elapsed,
        'finish_reason': choice.finish_reason if choice else '',
        'model_version': response.model,
        'prompt_tokens': getattr(usage, 'prompt_tokens', None),
        'answer_tokens': getattr(usage, 'completion_tokens', None),
        'thinking_tokens': getattr(token_details, 'reasoning_tokens', None),
        'total_tokens': getattr(usage, 'total_tokens', None),
    }

def get_groq_key():
    key = os.getenv('GROQ_API_KEY')
    if key:
        return key
    if Path('/kaggle/working').is_dir():
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret('GROQ_API_KEY')
        except Exception:
            return None
    return None

GROQ_API_KEY = get_groq_key()
RUN_LLM = bool(GROQ_API_KEY)
print('Groq answers enabled' if RUN_LLM else 'Groq answers skipped: add GROQ_API_KEY to run them')

## 7. Exploratory comparisons and retrieval timing

These questions include explicit exceptions and scope restrictions. Rank promotions are only candidate repairs: review the exported excerpts and answers against the printed rules before calling any query a baseline failure. Timings use one warm-up and five measured retrieval runs per query.

In [ ]:
import json
import pandas as pd

EXPLORATORY_QUERIES = [
    'Can a bird card substitute for another suit when you must discard cards?',
    'Can a card of another suit substitute when a rule specifically asks for a bird card?',
    'Can a player move along a river as though it were a path?',
    'Can the Marquise use Field Hospitals on Riverfolk mercenaries?',
    'Can the Knaves of the Deepwood and the Vagabond be played together?',
    'Can two Corvid plot tokens be placed in the same clearing?',
    'Does the attacker get an extra hit when the defender has no warriors?',
    'Which rule wins if a card conflicts with the Law?',
    'After buying Riverboats, may I keep treating rivers as paths on another player’s turn?',
    'If I buy Riverfolk mercenaries, can I move those warriors or count them toward dominance?',
    'Can another faction move pieces into the clearing with the Marquise keep, and can the keep be replaced after removal?',
    'Can a Vagabond form a coalition with a Hostile faction, and what happens to its relationship marker?',
    'Can a hireling use its controller’s Boat Builders effect to move on rivers?',
    'Can the controller include its own warriors in a hireling’s Move action?',
    'Does a hireling’s controller score points when the hireling removes an enemy building?',
]

# Source-page hints guide manual review; they do not label retrieval quality.
RULE_PAGE_HINTS = dict(zip(EXPLORATORY_QUERIES[8:], [
    '12 (11.2.7.II)', '12 (11.2.7.III)', '5 (6.2.2)',
    '10 (9.2.9.d)', '26 (H.3.5)', '26 (H.3.4)', '26 (H.3.6)',
]))

compare_retrieval(EXPLORATORY_QUERIES[0])  # warm up both encoders and FAISS
TIMING_REPEATS = 5
comparisons = []
timing_rows = []
for query in EXPLORATORY_QUERIES:
    for repeat in range(1, TIMING_REPEATS + 1):
        measured = compare_retrieval(query)
        timing_rows.append({'query': query, 'repeat': repeat, **measured['timing']})
        if repeat == 1:
            comparisons.append(measured)
        else:
            assert [c['chunk_id'] for c in measured['baseline_top10']] == [c['chunk_id'] for c in comparisons[-1]['baseline_top10']]
comparison_rows = []
for comparison in comparisons:
    new_ranks = {c['chunk_id']: c['reranked_rank'] for c in comparison['reranked_top10']}
    for candidate in comparison['baseline_top10']:
        comparison_rows.append({
            'query': comparison['query'], 'chunk_id': candidate['chunk_id'],
            'rule_page_hint': RULE_PAGE_HINTS.get(comparison['query'], ''),
            'baseline_rank': candidate['baseline_rank'],
            'reranked_rank': new_ranks[candidate['chunk_id']],
            'pages': f"{candidate['page_start']}-{candidate['page_end']}",
            'rule_refs': ', '.join(candidate['rule_refs']),
            'cosine_score': candidate['cosine_score'],
            'cross_score': next(c['cross_score'] for c in comparison['reranked_top10']
                                if c['chunk_id'] == candidate['chunk_id']),
            'baseline_top3': candidate['baseline_rank'] <= 3,
            'reranked_top3': new_ranks[candidate['chunk_id']] <= 3,
        })
rank_table = pd.DataFrame(comparison_rows)
timing_table = pd.DataFrame(timing_rows)
timing_table['baseline_retrieval_s'] = timing_table['query_embedding_s'] + timing_table['vector_search_s']
timing_table['reranked_retrieval_s'] = timing_table['baseline_retrieval_s'] + timing_table['cross_encoder_s']
timing_summary = timing_table.groupby('query', as_index=False, sort=False)[
    ['query_embedding_s', 'vector_search_s', 'cross_encoder_s',
     'baseline_retrieval_s', 'reranked_retrieval_s']].median()
print(rank_table.to_string(index=False, max_rows=20))
promotions = rank_table[(rank_table['baseline_rank'] > 3) & (rank_table['reranked_rank'] <= 3)]
print('\nChunks promoted from baseline ranks 4–10 into reranked top 3:')
print(promotions[['query', 'chunk_id', 'baseline_rank', 'reranked_rank', 'pages', 'rule_refs']].to_string(index=False))
print('\nRetrieval timing (seconds):')
print(timing_table[['baseline_retrieval_s', 'reranked_retrieval_s']].median().to_string())

OUTPUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('outputs')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rank_table.to_csv(OUTPUT_DIR / 'rank_shifts.csv', index=False)
timing_table.to_csv(OUTPUT_DIR / 'retrieval_timing.csv', index=False)
timing_summary.to_csv(OUTPUT_DIR / 'retrieval_timing_summary.csv', index=False)
evidence_table = rank_table.merge(
    pd.DataFrame([{'query': c['query'], 'chunk_id': item['chunk_id'], 'text': item['text']}
                  for c in comparisons for item in c['baseline_top10']]),
    on=['query', 'chunk_id'], validate='one_to_one')
evidence_table.to_csv(OUTPUT_DIR / 'retrieved_evidence.csv', index=False)
pd.DataFrame(chunks).to_csv(OUTPUT_DIR / 'corpus_chunks.csv', index=False)
run_config = {
    'pdf_sha256': actual_sha256, 'pdf_pages': len(page_text),
    'rule_pages': [min(RULE_PAGES), max(RULE_PAGES)],
    'bi_encoder': BI_ENCODER_NAME, 'cross_encoder': CROSS_ENCODER_NAME,
    'chunk_count': len(chunks), 'target_words': TARGET_WORDS,
    'max_words': MAX_WORDS, 'overlap_words': OVERLAP_WORDS,
    'timing_repeats': TIMING_REPEATS, 'llm_provider': 'Groq',
    'llm_model': GROQ_MODEL, 'llm_max_output_tokens': GROQ_MAX_OUTPUT_TOKENS,
    'llm_reasoning_effort': 'none', 'llm_temperature': 0.2,
    'bi_model_load_s': bi_model_load_seconds,
    'cross_model_load_s': cross_model_load_seconds, 'index_s': index_seconds,
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
print(f'Retrieval and evidence exports saved to {OUTPUT_DIR.resolve()}')

print('Run the final recovery cell to generate or resume answers.')


## 8. Resume answers and screen additional failure cases

In an existing Kaggle session, copy the next code cell to the end and run only that cell. It reads the existing CSV evidence, preserves the original `answers.csv`, and writes `answers_completed.csv`, skipping finished Groq answers. Short 429 limits are retried using the server's wait time. With the retrieval models still loaded, it also screens extra queries against source-rule anchors and generates up to three candidate pairs from distinct rule families. These are unverified candidates until you inspect both answers and all evidence; this cell does not write the required student-authored report.

In [ ]:
# Run this cell in the existing Kaggle session to resume the exported results.
import csv
import json
import os
import re
from collections import defaultdict
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
from time import perf_counter, sleep
from groq import Groq, RateLimitError

RECOVERY_DIR = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path('outputs')
RECOVERY_MODEL = 'qwen/qwen3.8-27b'
RECOVERY_MAX_TOKENS = 1024
RECOVERY_INTERVAL_S = 20
MAX_RATE_RETRIES = 5
MAX_NEW_REPAIR_PAIRS = 3  # Different rule families; screened before spending API calls.
RECOVERY_PROMPT = (
    'Answer the question using only the document excerpts below. Include relevant restrictions, '
    'exceptions, and conditions. If the excerpts do not contain enough information, say so. '
    'Cite supporting chunk IDs and page numbers.\n\nQuestion: {query}\n\nDocument excerpts:\n{context}'
)

def recovery_read(path):
    with path.open(encoding='utf-8-sig', newline='') as handle:
        return list(csv.DictReader(handle))

def recovery_save(path, records):
    if not records:
        return
    fields = list(dict.fromkeys(key for row in records for key in row))
    with path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=fields)
        writer.writeheader()
        writer.writerows(records)

def recovery_context(selected):
    return '\n\n'.join(
        f"[{c['chunk_id']}, pages {c['page_start']}-{c['page_end']}]\n{c['text']}"
        for c in selected
    )

def recovery_fingerprint(query, selected):
    payload = [RECOVERY_MODEL, RECOVERY_MAX_TOKENS, 0.2, 'none',
               RECOVERY_PROMPT.format(query=query, context=recovery_context(selected))]
    return sha256(json.dumps(payload, ensure_ascii=False).encode('utf-8')).hexdigest()

def recovery_complete(row, pipeline, fingerprint, selected):
    text = str(row.get(f'{pipeline}_answer', '') or '')
    if (not text.strip() or row.get(f'{pipeline}_finish_reason') != 'stop'
            or row.get(f'{pipeline}_model_version') != RECOVERY_MODEL
            or row.get(f'{pipeline}_error')):
        return False
    saved_hash = row.get(f'{pipeline}_context_sha256')
    if saved_hash:
        return saved_hash == fingerprint
    # The first Groq export predates hashes. Adopt it only when its cited chunks
    # belong to this saved context and its model/finish status match above.
    citations = set(re.findall(r'chunk_\d{4}', text))
    return bool(citations) and citations <= {c['chunk_id'] for c in selected}

_recovery_last_start = float('-inf')

def recovery_generate(client, query, selected):
    global _recovery_last_start
    waited = 0.0
    prompt = RECOVERY_PROMPT.format(query=query, context=recovery_context(selected))
    for attempt in range(1, MAX_RATE_RETRIES + 2):
        delay = max(0.0, RECOVERY_INTERVAL_S - (perf_counter() - _recovery_last_start))
        if delay:
            sleep(delay)
            waited += delay
        _recovery_last_start = perf_counter()
        try:
            response = client.chat.completions.create(
                model=RECOVERY_MODEL, messages=[{'role': 'user', 'content': prompt}],
                reasoning_effort='none', temperature=0.2,
                max_completion_tokens=RECOVERY_MAX_TOKENS,
            )
        except RateLimitError as exc:
            if attempt > MAX_RATE_RETRIES:
                raise
            headers = getattr(getattr(exc, 'response', None), 'headers', {})
            retry_after = headers.get('retry-after')
            try:
                delay = float(retry_after)
            except (TypeError, ValueError):
                match = re.search(r'try again in\s+([\d.]+)\s*(ms|s)', str(exc), re.I)
                delay = float(match.group(1)) / (1000 if match.group(2).lower() == 'ms' else 1) if match else 30
            delay = max(1.0, delay + 1)
            if delay > 120:
                raise RuntimeError('Quota requires a longer wait. Results are saved; rerun this cell later.') from exc
            print(f'Groq rate limit: waiting {delay:.1f}s before retry {attempt}/{MAX_RATE_RETRIES}')
            remaining = delay
            while remaining > 0:
                sleep(min(remaining, 60))
                remaining -= min(remaining, 60)
            waited += delay
            continue
        latency = perf_counter() - _recovery_last_start
        choice = response.choices[0] if response.choices else None
        usage = response.usage
        return {
            'answer': (choice.message.content or '') if choice else '',
            'finish_reason': choice.finish_reason if choice else '',
            'model_version': response.model, 'llm_s': latency,
            'prompt_tokens': getattr(usage, 'prompt_tokens', None),
            'answer_tokens': getattr(usage, 'completion_tokens', None),
            'thinking_tokens': getattr(getattr(usage, 'completion_tokens_details', None), 'reasoning_tokens', None),
            'total_tokens': getattr(usage, 'total_tokens', None),
            'request_attempts': attempt, 'wait_s': waited,
        }

def recovery_resume(client, items, output_path, seed_path=None):
    saved = {}
    for path in [seed_path, output_path]:
        if path is not None and path.is_file():
            saved.update({row['query']: row for row in recovery_read(path)})
    for item in items:
        query = item['query']
        row = saved.setdefault(query, {'query': query})
        for pipeline in ['baseline', 'reranked']:
            selected = item[f'{pipeline}_top3']
            fingerprint = recovery_fingerprint(query, selected)
            if recovery_complete(row, pipeline, fingerprint, selected):
                row[f'{pipeline}_context_sha256'] = fingerprint
                print('Already complete:', pipeline, query)
                recovery_save(output_path, list(saved.values()))
                continue
            try:
                details = recovery_generate(client, query, selected)
                row.update({f'{pipeline}_{key}': value for key, value in details.items()})
                row[f'{pipeline}_error'] = ''
                row[f'{pipeline}_context_sha256'] = fingerprint
            except Exception as exc:
                row[f'{pipeline}_error'] = f'{type(exc).__name__}: {exc}'
                recovery_save(output_path, list(saved.values()))
                raise RuntimeError(f'Answer run paused at {pipeline}: {query}. Completed answers are saved; rerun this cell to resume.') from exc
            recovery_save(output_path, list(saved.values()))
            print('Saved:', pipeline, row[f'{pipeline}_finish_reason'], query)
            if not recovery_complete(row, pipeline, fingerprint, selected):
                print('Answer needs review: finish reason is not stop or text is empty.')
    return saved

# Reconstruct exactly the original contexts from the exported evidence.
evidence_path = RECOVERY_DIR / 'retrieved_evidence.csv'
if not evidence_path.is_file():
    raise FileNotFoundError(f'Keep the previous output CSVs in {RECOVERY_DIR}; missing {evidence_path.name}')
recovery_evidence = recovery_read(evidence_path)
by_query = defaultdict(list)
for row in recovery_evidence:
    first, last = row['pages'].split('-')
    by_query[row['query']].append({
        **row, 'page_start': int(first), 'page_end': int(last),
        'baseline_rank': int(row['baseline_rank']), 'reranked_rank': int(row['reranked_rank']),
    })
recovery_items = [
    {'query': query,
     'baseline_top3': sorted(rows, key=lambda x: x['baseline_rank'])[:3],
     'reranked_top3': sorted(rows, key=lambda x: x['reranked_rank'])[:3]}
    for query, rows in by_query.items()
]

recovery_key = os.getenv('GROQ_API_KEY') or globals().get('GROQ_API_KEY')
if not recovery_key and Path('/kaggle/working').is_dir():
    from kaggle_secrets import UserSecretsClient
    recovery_key = UserSecretsClient().get_secret('GROQ_API_KEY')
if not recovery_key:
    raise RuntimeError('Enable the GROQ_API_KEY Kaggle Secret, then rerun this cell.')

with Groq(api_key=recovery_key, max_retries=0) as recovery_client:
    recovered_answers = recovery_resume(
        recovery_client, recovery_items, RECOVERY_DIR / 'answers_completed.csv',
        RECOVERY_DIR / 'answers.csv',
    )

# Each group tests a distinct rule. Gold text is used ONLY for screening and
# review, never added to the context or prompt of either answer pipeline.
STRESS_RULES = [
    ('slip-snare', 'This move ignores all effects that prevent movement out of a clearing',
     'Slip can leave the snare clearing; the explicit movement exception applies.',
     ['A Corvid snare prevents pieces leaving its clearing. Can the Vagabond still Slip out during Birdsong?',
      'Can the Vagabond escape a snare without exhausting boots?',
      'Does a face-up snare stop the Vagabond from leaving with Slip?']),
    ('raid-exposure', 'if this removes a Raid token, it does not place warriors.',
     'Correct Exposure removes the raid without placing warriors.',
     ['If I correctly guess and remove a Corvid raid with Exposure, do Corvid warriors appear?',
      'Does destroying a face-down raid by guessing its type trigger the raid warrior placement?',
      'When I expose a raid with a matching card, does removing it recruit crows?']),
    ('coalition-hostile', 'If you do, move their relationship marker to the Indifferent space.',
     'A coalition with a Hostile faction is allowed; its relationship marker becomes Indifferent, subject to normal coalition eligibility.',
     ['Does forming a coalition with an enemy reset the Vagabond relationship from Hostile?',
      'Can the Vagabond join a Hostile faction and move its marker to Indifferent?',
      'When the Vagabond allies through dominance with a Hostile player, what happens to the relationship?']),
    ('winter-river', 'The river divides forests (as printed paths do).',
     'On the Winter map the river divides forests, overriding the ordinary map rule.',
     ['On the Winter map, can a forest extend across the river?',
      'Do rivers separate forests when playing the winter board?',
      'Is the general rule that rivers do not divide forests also true on the Winter map?']),
    ('frog-discard', 'discard frog cards to the Pond, not the Lost Souls.',
     'Discarded frog cards go to the Pond even when the Lizard Cult is playing.',
     ['With the Lizard Cult playing, does a discarded frog card enter Lost Souls?',
      'Do frog discards affect the Cult outcast by entering the Lost Souls pile?']),
    ('late-exposure', 'You cannot use Exposure once you have begun the last step of your turn.',
     'Exposure cannot be used after beginning the final step of the turn.',
     ['After starting my final draw-and-discard step, may I still expose a Corvid plot?',
      'Can I guess a face-down plot after beginning the last step of Evening?']),
    ('warlord-banish', 'The warlord cannot take hits from banish',
     'The warlord cannot take banish hits because it cannot be moved outside its own turn.',
     ['Can the Twilight Council banish the Lord of the Hundreds warlord outside his turn?',
      'Does the Council force the warlord to move when he takes a banish hit?']),
    ('hostile-aid', 'You cannot move a relationship marker out of the Hostile box by using Aid',
     'Aid may take crafted items but cannot remove Hostile status.',
     ['Can repeated Aid repair the Vagabond relationship with a Hostile faction?',
      'If the Vagabond gives enough cards to a Hostile faction for items, does it become friendly again?']),
]

def recovery_normalize(text):
    return re.sub(r'\s+', ' ', text).casefold().strip()

stress_items, stress_screening, stress_evidence, stress_timings = [], [], [], []
stress_possible = all(name in globals() for name in ['compare_retrieval', 'chunks', 'bi_encoder', 'cross_encoder', 'vector_index'])
if stress_possible:
    for rule_id, gold_text, expected, queries in STRESS_RULES:
        gold_ids = {c['chunk_id'] for c in chunks
                    if recovery_normalize(gold_text) in recovery_normalize(c['text'])}
        if not gold_ids:
            print('Source anchor not found; skipping rule:', rule_id)
            continue
        for query in queries:
            item = compare_retrieval(query)  # Same ordinary top 10; no gold injection.
            baseline_gold = [c['baseline_rank'] for c in item['baseline_top10'] if c['chunk_id'] in gold_ids]
            reranked_gold = [c['reranked_rank'] for c in item['reranked_top10'] if c['chunk_id'] in gold_ids]
            b = min(baseline_gold, default=11)
            r = min(reranked_gold, default=11)
            candidate = 4 <= b <= 10 and r <= 3
            stress_screening.append({
                'rule_id': rule_id, 'query': query, 'gold_text': gold_text,
                'expected_answer_for_review': expected,
                'gold_chunk_ids': ', '.join(sorted(gold_ids)),
                'best_baseline_gold_rank': b if b <= 10 else 'outside_top10',
                'best_reranked_gold_rank': r if r <= 10 else 'outside_top10',
                'candidate_repair': candidate,
                'review_status': 'UNVERIFIED: inspect all context and both answers',
            })
            new_ranks = {c['chunk_id']: c for c in item['reranked_top10']}
            for c in item['baseline_top10']:
                stress_evidence.append({
                    'rule_id': rule_id, 'query': query, 'chunk_id': c['chunk_id'],
                    'baseline_rank': c['baseline_rank'],
                    'reranked_rank': new_ranks[c['chunk_id']]['reranked_rank'],
                    'cosine_score': c['cosine_score'],
                    'cross_score': new_ranks[c['chunk_id']]['cross_score'],
                    'pages': f"{c['page_start']}-{c['page_end']}", 'text': c['text'],
                    'contains_gold_anchor': c['chunk_id'] in gold_ids,
                })
            if candidate and rule_id not in {x['rule_id'] for x in stress_items}:
                stress_items.append({**item, 'rule_id': rule_id})
    recovery_save(RECOVERY_DIR / 'stress_screening.csv', stress_screening)
    recovery_save(RECOVERY_DIR / 'stress_evidence.csv', stress_evidence)
    stress_items = stress_items[:MAX_NEW_REPAIR_PAIRS]
    print('Distinct rule families with candidate retrieval repairs:', len(stress_items))
    for item in stress_items:
        print('Candidate:', item['rule_id'], item['query'])
        for repeat in range(1, 6):
            timing = compare_retrieval(item['query'])['timing']
            baseline_s = timing['query_embedding_s'] + timing['vector_search_s']
            stress_timings.append({
                'query': item['query'], 'repeat': repeat, **timing,
                'baseline_retrieval_s': baseline_s,
                'reranked_retrieval_s': baseline_s + timing['cross_encoder_s'],
            })
    recovery_save(RECOVERY_DIR / 'stress_timing.csv', stress_timings)
    if stress_items:
        with Groq(api_key=recovery_key, max_retries=0) as recovery_client:
            recovery_resume(recovery_client, stress_items, RECOVERY_DIR / 'stress_answers.csv')
    else:
        print('No new anchor-based repair found. More query exploration is required; do not claim two wins.')
else:
    print('Original answers recovered. To search extra cases, load the retrieval models/index first (sections 0-6).')

complete_pairs = sum(
    all(recovery_complete(recovered_answers[item['query']], p,
                          recovery_fingerprint(item['query'], item[f'{p}_top3']), item[f'{p}_top3'])
        for p in ['baseline', 'reranked'])
    for item in recovery_items
)
(RECOVERY_DIR / 'completion_run.json').write_text(json.dumps({
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'model': RECOVERY_MODEL,
    'input_evidence_sha256': sha256(evidence_path.read_bytes()).hexdigest(),
    'original_complete_pairs': complete_pairs, 'original_total_pairs': len(recovery_items),
    'extra_candidate_pairs': len(stress_items),
    'failure_cases_verified': False,
}, indent=2), encoding='utf-8')
print(f'Original answer pairs complete: {complete_pairs}/{len(recovery_items)}')
print('Download answers_completed.csv and any stress_*.csv files.')
print('Candidate repairs still require manual review. The student-written report and GitHub submission remain separate.')
